In [ ]:
import pandas as pd
import torch
import os
from transformers import BertTokenizer


# Dataset reading

In [ ]:

# Charger les fichiers CSV
file_d1 = "/kaggle/input/dataset-efw-1/D1_efw_dataset.csv"
file_d2 = "/kaggle/input/dataset-efw-1/D2_efw_dataset.csv"

df_d1 = pd.read_csv(file_d1, sep=",")  # Adapter le séparateur si nécessaire
df_d2 = pd.read_csv(file_d2, sep=",")



In [ ]:
df_d1.head()

In [ ]:
# Définition du répertoire contenant les fichiers CSV
repertoire_donnees = "/kaggle/input/stmf-dataset/"

# Liste pour stocker les noms de fichiers et les DataFrames
noms_fichiers = []
dataframes = []

# Parcours du répertoire et chargement des fichiers CSV
for fichier in sorted(os.listdir(repertoire_donnees)):  # Tri pour garantir l'ordre
    if fichier.endswith(".csv"):  # Vérifie si c'est un fichier CSV
        chemin_fichier = os.path.join(repertoire_donnees, fichier)
        df = pd.read_csv(chemin_fichier, sep=",")  # Charger le fichier en DataFrame
        
        # Ajouter à la liste
        noms_fichiers.append(fichier)
        dataframes.append(df)

# Affichage des résultats
print("📂 Fichiers chargés :", noms_fichiers)
print(f"📊 Nombre de DataFrames chargés : {len(dataframes)}")

In [ ]:

# Exemple d'accès : Premier fichier et son DataFrame
if dataframes:
    print("\n🔹 Premier fichier chargé :", noms_fichiers[0])
    print(dataframes[0].head())  # Afficher les 5 premières lignes


In [ ]:
# Fusionner tous les DataFrames en un seul
df_combined = pd.concat(dataframes, ignore_index=True)

# Vérification des colonnes
print("📊 Colonnes du dataset fusionné :", df_combined.columns)
print(f"📈 Nombre total de lignes après fusion : {df_combined.shape[0]}")


In [ ]:
# # Fusionner les deux datasets pour augmenter la taille
# df_combined = pd.concat([df_d1, df_d2], ignore_index=True)

# # Vérification des colonnes
# print("Colonnes du dataset :", df_combined.columns)



In [ ]:
print("Colonnes disponibles dans df_combined :", df_combined.columns)


In [ ]:
# ---- 1️⃣ Transformation : remplacer () par | ----
df_combined["patterns_transformed"] = (
    df_combined["patterns"]
    .str.replace(r"[()]", "|", regex=True)  # Remplacement
    .str.replace(r"\|+", "|", regex=True)   # Suppression des doublons
    .str.strip("|")  # Suppression des séparateurs en début/fin de chaîne
)

In [ ]:
print("Colonnes disponibles dans df_combined :", df_combined.columns)

In [ ]:
df_combined.head()

In [ ]:
# # Initialisation du tokenizer
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# # Fonction pour tokenizer les motifs
# def tokenize_patterns(pattern):
#     return tokenizer(pattern, padding="max_length", truncation=True, return_tensors="pt")

# # Appliquer la tokenisation sur les motifs
# df_combined["tokenized_patterns"] = df_combined["patterns"].apply(lambda x: tokenize_patterns(x)["input_ids"].squeeze(0))

# # Convertir les mesures en tenseurs
# gs_tensor = torch.tensor(df_combined["Supp X"].values).float()
# wgs_tensor = torch.tensor(df_combined["Supp Y"].values).float()
# acc_tensor = torch.tensor(df_combined["Supp Z"].values).float()

# # Fusionner les embeddings des motifs et les mesures en entrée du modèle
# dataset_tensors = {
#     "patterns": df_combined["tokenized_patterns"].tolist(),
#     "gs": gs_tensor,
#     "wgs": wgs_tensor,
#     "acc": acc_tensor
# }

# # Sauvegarder le dataset préparé
# torch.save(dataset_tensors, "dataset_transformer_input.pt")

# print("Dataset transformé et sauvegardé sous dataset_transformer_input.pt: /kaggle/working/dataset_transformer_input.pt")


In [ ]:
# Extraire les motifs uniques pour éviter que le tokenizer sépare les nombres et "+"
# Nettoyage et transformation des motifs
def clean_pattern(pattern):
    """ Nettoie les motifs en supprimant les espaces inutiles autour des séparateurs """
    return " | ".join([p.strip() for p in pattern.split("|")])  # Nettoie chaque motif séparé par |

df_combined["patterns_transformed"] = df_combined["patterns_transformed"].apply(clean_pattern)

# Extraire tous les motifs uniques pour éviter les erreurs de découpage
motifs_uniques = set()
for motif in df_combined["patterns_transformed"]:
    motifs_uniques.update(motif.split(" | "))  # Ajoute chaque motif individuel

# Initialisation du tokenizer et ajout des nouveaux tokens
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.add_tokens(list(motifs_uniques))  # Ajouter les motifs au vocabulaire

# Fonction améliorée pour la tokenisation
def tokenize_patterns(pattern):
    """ Tokenise le motif tout en vérifiant que les tokens sont bien reconnus """
    tokenized = tokenizer(pattern, padding="max_length", truncation=True, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'][0].tolist())
    
    # Vérification si le motif est bien tokenisé
    print(f"🔍 Motif original : {pattern}")
    print(f"📝 Tokens générés : {tokens}\n")
    
    return tokenized["input_ids"].squeeze(0)

# Appliquer la tokenisation avec logs
df_combined["tokenized_patterns"] = df_combined["patterns_transformed"].apply(tokenize_patterns)

# Convertir les colonnes numériques en tenseurs
gs_tensor = torch.tensor(df_combined["Supp X"].values).float()
wgs_tensor = torch.tensor(df_combined["Supp Y"].values).float()
acc_tensor = torch.tensor(df_combined["Supp Z"].values).float()

# Fusionner les embeddings des motifs et les mesures en entrée du modèle
dataset_tensors = {
    "patterns": df_combined["tokenized_patterns"].tolist(),
    "gs": gs_tensor,
    "wgs": wgs_tensor,
    "acc": acc_tensor
}

# Sauvegarder le dataset préparé
torch.save(dataset_tensors, "dataset_transformer_input.pt")

# Sauvegarder le tokenizer mis à jour pour réutilisation
tokenizer.save_pretrained("tokenizer_custom")

print("✅ Dataset transformé et sauvegardé sous dataset_transformer_input.pt")
print("✅ Tokenizer personnalisé sauvegardé sous tokenizer_custom/")

# Explorer le dataset PyTorch

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.tokenize("( 1+3+ )"))  # Vérifiez comment il coupe les tokens


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Désactiver la séparation automatique des caractères spéciaux
tokenizer.add_tokens(["(1+3+)", "(3+)(0+3+)"])

print(tokenizer.tokenize("(1+3+)"))  # Vérifier le découpage


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Exemple de données
data = {
    "patterns": [
        "(1+3+)", "(3+)(0+3+)", "(3+)(1+3+)",
        "(1+)(3+)(0+3+)", "(1+)(3+)(1+3+)",
        "(0+)(3+)(0+3+)", "(0+)(3+)(1+3+)"
    ]
}
df = pd.DataFrame(data)

# ---- 1️⃣ Transformation : remplacer () par | ----
df["patterns_transformed"] = (
    df["patterns"]
    .str.replace(r"[()]", "|", regex=True)  # Remplacement
    .str.replace(r"\|+", "|", regex=True)   # Suppression des doublons
    .str.strip("|")  # Suppression des séparateurs en début/fin de chaîne
)
# ---- 2️⃣ Tokenizer personnalisé ----
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
df["tokenized"] = df["patterns_transformed"].apply(lambda x: tokenizer.tokenize(x))

# ---- 3️⃣ Affichage des résultats ----
print("📌 Données originales")
print(df[["patterns"]])

print("\n🔹 Représentation avec | au lieu des parenthèses")
print(df[["patterns_transformed"]])

print("\n🔹 Tokenization (BERT)")
print(df[["tokenized"]])


In [ ]:

# Charger le dataset sauvegardé
dataset_path = "/kaggle/working/dataset_transformer_input.pt"  # Adapter le chemin si nécessaire
dataset_tensors = torch.load(dataset_path)

# Vérifier les clés du dataset
print("🔍 Clés disponibles dans le dataset :", dataset_tensors.keys())

# Afficher un aperçu des données
print("\n📝 Exemple de données tokenisées :", dataset_tensors["patterns"][:3])  # Afficher 3 exemples
print("\n📊 Valeurs GS :", dataset_tensors["gs"][:5])  # Afficher 5 premières valeurs GS
print("\n📊 Valeurs WGS :", dataset_tensors["wgs"][:5])  # Afficher 5 premières valeurs WGS
print("\n📊 Valeurs ACC :", dataset_tensors["acc"][:5])  # Afficher 5 premières valeurs ACC)


# Training Model

## Intégration au Modèle de Machine Learning

Maintenant que la tokenisation est corrigée et que les motifs sont bien représentés dans le vocabulaire, voici les prochaines étapes :
📌 Étapes suivantes

1️⃣ Préparer les Tenseurs pour le Modèle

    Vérifier la taille des tenseurs et s'assurer qu'ils sont compatibles avec le modèle.
    Normaliser les valeurs si nécessaire.

2️⃣ Définir le Modèle Transformer

    Charger un modèle BERT ou Transformer personnalisé.
    Adapter la couche d'entrée pour traiter les tokens des motifs + les mesures numériques (Supp X, Supp Y, Supp Z).

3️⃣ Entraînement du Modèle

    Diviser le dataset en train/test.
    Définir une fonction de loss et un optimizer.
    Lancer l'entraînement et suivre la courbe de précision / perte.

4️⃣ Évaluation et Amélioration

    Tester le modèle sur des motifs non vus.
    Analyser les erreurs et ajuster le modèle si besoin (ex: data augmentation).

5️⃣ Déploiement et Utilisation

    Sauvegarder le modèle (.pt ou .pkl).
    Créer une API Flask/FastAPI pour l’utiliser en inférence sur de nouveaux motifs.



## Étape 1 : Chargement des Données et Prétraitement

Vérifions d'abord que les données sont bien prêtes et utilisables par le modèle.
1️⃣ Charger le dataset transformé

In [ ]:
#import torch

# Charger le dataset transformé
dataset_path = "/kaggle/working/dataset_transformer_input.pt"
dataset_tensors = torch.load(dataset_path)

# Vérifier les clés et formats
print("Clés disponibles :", dataset_tensors.keys())
print("Exemple de tokenized_patterns :", dataset_tensors["patterns"][0])
print("Shape des tenseurs gs, wgs, acc :", dataset_tensors["gs"].shape, dataset_tensors["wgs"].shape, dataset_tensors["acc"].shape)


## 2️⃣ Convertir les données en un DataLoader

Nous devons créer un Dataset PyTorch et un DataLoader pour charger les données efficacement.

In [ ]:
# from torch.utils.data import Dataset, DataLoader

# class MotifDataset(Dataset):
#     def __init__(self, dataset):
#         self.patterns = dataset["patterns"]
#         self.gs = dataset["gs"]
#         self.wgs = dataset["wgs"]
#         self.acc = dataset["acc"]

#     def __len__(self):
#         return len(self.gs)

#     def __getitem__(self, idx):
#         return {
#             "patterns": torch.tensor(self.patterns[idx]),  # Séquence tokenisée
#             "gs": self.gs[idx],  # Valeur Supp X
#             "wgs": self.wgs[idx],  # Valeur Supp Y
#             "acc": self.acc[idx]  # Valeur Supp Z
#         }

# # Création du dataset
# motif_dataset = MotifDataset(dataset_tensors)

# # Création du DataLoader
# dataloader = DataLoader(motif_dataset, batch_size=16, shuffle=True)

# # Tester un batch
# for batch in dataloader:
#     print("Exemple de batch :", batch)
#     break


In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import torch

class MotifDataset(Dataset):
    def __init__(self, dataset, normalize=True):
        self.patterns = dataset["patterns"]
        self.gs = dataset["gs"]
        self.wgs = dataset["wgs"]
        self.acc = dataset["acc"]
        
        # Normalisation des features numériques
        if normalize:
            scaler = MinMaxScaler()
            self.gs = scaler.fit_transform(self.gs.reshape(-1, 1)).flatten()
            self.wgs = scaler.fit_transform(self.wgs.reshape(-1, 1)).flatten()
            self.acc = scaler.fit_transform(self.acc.reshape(-1, 1)).flatten()

    def __len__(self):
        return len(self.gs)

    def __getitem__(self, idx):
        return {
            "patterns": torch.tensor(self.patterns[idx], dtype=torch.long),  # Séquence tokenisée long remplacer flaot 32
            "gs": torch.tensor(self.gs[idx], dtype=torch.float32),  # Valeur normalisée de Supp X
            "wgs": torch.tensor(self.wgs[idx], dtype=torch.float32),  # Valeur normalisée de Supp Y
            "acc": torch.tensor(self.acc[idx], dtype=torch.float32)  # Valeur normalisée de Supp Z
        }

# Création du dataset avec normalisation
motif_dataset = MotifDataset(dataset_tensors, normalize=True)

# Création du DataLoader
dataloader = DataLoader(motif_dataset, batch_size=16, shuffle=True)

# Tester un batch
for batch in dataloader:
    print("Exemple de batch normalisé :", batch)
    break


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extraction des données sous forme de liste
all_gs = [x["gs"].item() for x in motif_dataset]
all_wgs = [x["wgs"].item() for x in motif_dataset]
all_acc = [x["acc"].item() for x in motif_dataset]

# Création des subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramme de gs (Supp X)
sns.histplot(all_gs, bins=20, kde=True, ax=axes[0], color="blue")
axes[0].set_title("Distribution de GS (Supp X)")

# Histogramme de wgs (Supp Y)
sns.histplot(all_wgs, bins=20, kde=True, ax=axes[1], color="green")
axes[1].set_title("Distribution de WGS (Supp Y)")

# Histogramme de acc (Supp Z)
sns.histplot(all_acc, bins=20, kde=True, ax=axes[2], color="red")
axes[2].set_title("Distribution de ACC (Supp Z)")

plt.show()


In [ ]:
import numpy as np

def detect_outliers(data):
    """ Détection des valeurs aberrantes avec la méthode de l'IQR """
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return [x for x in data if x < lower_bound or x > upper_bound]

outliers_gs = detect_outliers(all_gs)
outliers_wgs = detect_outliers(all_wgs)
outliers_acc = detect_outliers(all_acc)

print(f"📌 Outliers détectés - GS: {len(outliers_gs)}, WGS: {len(outliers_wgs)}, ACC: {len(outliers_acc)}")


In [ ]:
def replace_outliers(data):
    """ Remplace les valeurs aberrantes par la médiane """
    median = np.median(data)
    Q1, Q3 = np.percentile(data, [25, 75])
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return [x if lower_bound <= x <= upper_bound else median for x in data]

clean_gs = replace_outliers(all_gs)
clean_wgs = replace_outliers(all_wgs)
clean_acc = replace_outliers(all_acc)


In [ ]:
filtered_gs = [x for x in all_gs if x not in outliers_gs]
filtered_wgs = [x for x in all_wgs if x not in outliers_wgs]
filtered_acc = [x for x in all_acc if x not in outliers_acc]



## Étape 2 : Définition du Modèle Transformer

Nous allons créer un modèle utilisant une couche Transformer Encoder pour traiter les motifs et les fusionner avec les valeurs numériques.

### 1️⃣ Définir l'architecture
✅ Objectif : Construire un modèle Transformer qui prend en entrée les tokens des motifs + les valeurs numériques.

In [ ]:
import torch.nn as nn
import torch.optim as optim

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, num_layers=2, ff_dim=256, output_dim=1):
        super(TransformerClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)
        
        # Couche pour fusionner les valeurs numériques
        self.fc_numeric = nn.Linear(3, embed_dim)  # (gs, wgs, acc)
        
        # Couche finale de classification
        self.fc_out = nn.Linear(embed_dim, output_dim)
        self.sigmoid = nn.Sigmoid()  # Si classification binaire

    def forward(self, patterns, gs, wgs, acc):
        embedded_patterns = self.embedding(patterns)  # Convertir en vecteurs
        transformer_output = self.transformer_encoder(embedded_patterns)  # Passage dans le Transformer
        
        # Prendre uniquement le token [CLS] (premier token)
        cls_token = transformer_output[:, 0, :]

        # Fusionner avec les valeurs numériques
        numeric_features = torch.cat((gs.unsqueeze(1), wgs.unsqueeze(1), acc.unsqueeze(1)), dim=1)
        numeric_embedding = self.fc_numeric(numeric_features)

        # Fusion des deux
        combined = cls_token + numeric_embedding

        # Prédiction finale
        output = self.fc_out(combined)
        return self.sigmoid(output)

# Instanciation du modèle
vocab_size = 30600  # Taille du vocabulaire BERT (peut être réduit si personnalisé)
model = TransformerClassifier(vocab_size)

# Déplacement sur GPU si disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(model)


## 📌 Étape 3 : Entraînement du Modèle

### Nous allons entraîner notre modèle avec un optimiseur Adam et une Binary Cross Entropy Loss (BCE).
1️⃣ Définir la fonction de perte et l'optimiseur

In [ ]:
#criterion = nn.BCELoss()  # Pour classification binaire
def weighted_huber_loss(predictions, ground_truth):
    huber = nn.HuberLoss(delta=0.1, reduction='none')  # Ne pas moyenner immédiatement
    loss = huber(predictions, ground_truth)
    
    # Augmenter le poids des valeurs critiques (1.00 et 0.66)
    weights = torch.where((ground_truth == 1.00) | (ground_truth == 0.66), 2.0, 1.0)
    
    return (loss * weights).mean()  # Moyenne pondérée des erreurs

def weighted_loss(preds, targets):
    weights = torch.where(targets >= 0.83, 2.0, 1.0)  # Plus de poids aux valeurs élevées
    loss = criterion(preds, targets) * weights
    return loss.mean()


# criterion = weighted_huber_loss  # Utilisation de la nouvelle fonction
criterion = nn.SmoothL1Loss(beta=0.05)  # Plus robuste aux valeurs extrêmes # Moins de tolérance aux grosses erreurs


optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
print("Valeurs max des tokens :", max([max(seq) for seq in dataset_tensors["patterns"] if len(seq) > 0]))
print("Vocab size attendu :", vocab_size)


## ✅ Objectif : Entraîner le modèle sur plusieurs époques et suivre l'évolution de la perte.

In [ ]:
from tqdm import tqdm

num_epochs = 10  # Nombre d'époques
model.train()  # Mode entraînement

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in tqdm(dataloader):
        optimizer.zero_grad()
        
        # Charger les données
        patterns = batch["patterns"].to(device)
        gs = batch["gs"].to(device)
        wgs = batch["wgs"].to(device)
        acc = batch["acc"].to(device)
        # 🔥 Vérifier et définir `ground_truth`
        ground_truth = batch.get("ground_truth", batch["gs"]).to(device)  # 🔥 Si absent, prend `gs`
        

        # Forward pass
        outputs = model(patterns, gs, wgs, acc).squeeze()

        # Calcul de la perte
       # loss = criterion(outputs, gs)  # Exemple : prédire gs
        # loss = weighted_huber_loss(outputs, ground_truth)
        loss = weighted_loss(outputs, gs)
        
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}")

print("✅ Entraînement terminé !")


## 📌 Étape 4 : Évaluation

Une fois l'entraînement terminé, évaluons la performance du modèle.
### 1️⃣ Tester sur un batch de validation
✅ Objectif : Vérifier si le modèle fait des prédictions cohérentes.

In [ ]:
model.eval()
with torch.no_grad():
    for batch in dataloader:
        patterns = batch["patterns"].to(device)
        gs = batch["gs"].to(device)
        wgs = batch["wgs"].to(device)
        acc = batch["acc"].to(device)

        outputs = model(patterns, gs, wgs, acc).squeeze()
        print("Prédictions :", outputs)
        print("Valeurs réelles :", gs)  # Exemple
        break


## 📌 Étape 5 : Sauvegarde et Déploiement

### 1️⃣ Sauvegarde du modèle

In [ ]:
torch.save(model.state_dict(), "transformer_model.pth")
print("✅ Modèle sauvegardé sous transformer_model.pth")


### 2️⃣ Charger le modèle pour une future utilisation

In [ ]:
model.load_state_dict(torch.load("transformer_model.pth"))
model.eval()

## 